In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist

In [ ]:
# loading the dataset
(train_X, train_Y), (test_X, test_Y) = mnist.load_data()

# flatten the data from 28x28 to 784x1
train_X_flattened = train_X.reshape(train_X.shape[0], -1)
test_X_flattened = test_X.reshape(test_X.shape[0], -1)

# convert to pandas dataframe
train_df = pd.DataFrame(train_X_flattened)
test_df = pd.DataFrame(test_X_flattened)

# add labels to the dataframes to the end of the dataframes
train_df["label"] = train_Y
test_df["label"] = test_Y

In [ ]:
# transform it to numpy array
train_data = np.array(train_df)
test_data = np.array(test_df)

# shuffle the data
np.random.shuffle(train_data)
np.random.shuffle(test_data)

# split the data into features and labels
X_train = train_data[:, :-1] # all columns except the last one 
Y_train = train_data[:, -1] # only last column
X_test = test_data[:, :-1]
Y_test = test_data[:, -1]

In [ ]:
# initialize the weights and biases
def init_params():
    # first layer 784 input neurons (input layer), 64 output neurons (1. hidden layer)
    w1 = np.random.randn(784, 64) * 0.01
    b1 = np.zeros((1, 64))
    
    # second layer: 64 input neurons (1. hidden layer), 32 output neurons (2. hidden layer)
    w2 = np.random.randn(64, 32) * 0.01
    b2 = np.zeros((1, 32))
    
    # third layer: 32 input neurons (2. hidden layer), 10 output neurons (output layer)
    w3 = np.random.randn(32, 10) * 0.01
    b3 = np.zeros((1, 10))
    return w1, b1, w2, b2, w3, b3

# forward propagation with X as input and the weights and biases as parameters
def forward_prop(X, w1, b1, w2, b2, w3, b3):
    # first layer output
    Z1 = np.dot(X, w1) + b1
    A1 = relu(Z1)
    
    # second layer output
    Z2 = np.dot(A1, w2) + b2
    A2 = relu(Z2)
    
    # third layer output
    Z3 = np.dot(A2, w3) + b3
    # final output with softmax activation function
    A3 = softmax(Z3)
    return Z1, A1, Z2, A2, Z3, A3

# backward propagation with X as input, the weights and biases as parameters and the output of the forward propagation
def backward_prop(X, Z1, A1, Z2, A2, Z3, A3, Y, w2, w3):
    # number of samples
    m = Y.shape[0]
    
    one_hot_Y = one_hot(Y)
    
    # calculate the gradients for the output layer
    dZ3 = A3 - one_hot_Y
    dw3 = 1 / m * A2.T.dot(dZ3)
    db3 = 1 / m * np.sum(dZ3, axis=0, keepdims=True)
    
    dA2 = np.dot(dZ3, w3.T)
    dZ2 = dA2 * deriv_relu(Z2)
    dw2 = 1 / m * A1.T.dot(dZ2)
    db2 = 1 / m * np.sum(dZ2, axis=0, keepdims=True)
    
    dA1 = np.dot(dZ2, w2.T)
    dZ1 = dA1 * deriv_relu(Z1)
    dw1 = 1 / m * X.T.dot(dZ1)
    db1 = 1 / m * np.sum(dZ1, axis=0, keepdims=True)
    return dw1, db1, dw2, db2, dw3, db3

# update the weights and biases with the gradients and the learning rate alpha
def update_params(w1, b1, w2, b2, w3, b3, dw1, db1, dw2, db2, dw3, db3, alpha):
    w1 -= alpha * dw1
    b1 -= alpha * db1
    w2 -= alpha * dw2
    b2 -= alpha * db2
    w3 -= alpha * dw3
    b3 -= alpha * db3
    return w1, b1, w2, b2, w3, b3
    
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y
    
def relu(Z):
    return np.maximum(0, Z)

def deriv_relu(Z):
    return (Z > 0).astype(float)

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return expZ / np.sum(expZ, axis=1, keepdims=True)

def gradient_descent(X, Y, w1, b1, w2, b2, w3, b3, alpha):
    # first we do the forward propagation to get the output of the network
    Z1, A1, Z2, A2, Z3, A3 = forward_prop(X, w1, b1, w2, b2, w3, b3)
    # then we do the backward propagation to get the gradients of the weights and biases
    dw1, db1, dw2, db2, dw3, db3 = backward_prop(X, Z1, A1, Z2, A2, Z3, A3, Y, w2, w3)
    # finally we update the weights and biases with the gradients and the learning rate alpha
    w1, b1, w2, b2, w3, b3 = update_params(w1, b1, w2, b2, w3, b3, dw1, db1, dw2, db2, dw3, db3, alpha)
    return w1, b1, w2, b2, w3, b3 

In [ ]:
def calculate_accuracy(index, X, Y, w1, b1, w2, b2, w3, b3):
    # we do the forward propagation to get the output of the network
    _, _, _, _, _, A3 = forward_prop(X, w1, b1, w2, b2, w3, b3)
    # we get the predicted labels by taking the index of the maximum value in the output layer
    predictions = np.argmax(A3, axis=1)
    # we iterate through the predictions and compare them with the true labels to calculate the accuracy
    acc = np.mean(predictions == Y)
    return acc

def print_prediction(index, X, Y, prediction):
    image = X[index].reshape(28, 28)
    predictions = np.argmax(prediction, axis=1)
    plt.imshow(image)
    plt.title("Sample " + str(index))
    plt.axis("off")
    plt.show()
    
    print(f"Output of Neural Network: {np.round(prediction[index], 3).tolist()}")
    print(f"Predicted Label: {predictions[index]}")
    print(f"True Label: {Y[index]}")   

In [ ]:
def train(iterations, alpha, w1, b1, w2, b2, w3, b3):
    for i in range(iterations):
        w1, b1, w2, b2, w3, b3 = gradient_descent(X_train, Y_train, w1, b1, w2, b2, w3, b3, alpha)
        if ((i+1) % 5 == 0):
            print(f"Epoch {i + 1}/{iterations} completed")
            acc = calculate_accuracy(i, X_train, Y_train, w1, b1, w2, b2, w3, b3)
            print(f"Training Accuracy: {acc*100:.2f}%")
            
    return w1, b1, w2, b2, w3, b3
            
def test(index, X_test, Y_test, w1, b1, w2, b2, w3, b3):
    _, _, _, _, _, A3 = forward_prop(X_test, w1, b1, w2, b2, w3, b3)
    print_prediction(index, X_test, Y_test, A3)
    acc = calculate_accuracy(index, X_test, Y_test, w1, b1, w2, b2, w3, b3)
    print(f"Test Accuracy: {acc * 100:.2f}%")

In [ ]:
def train_model(iterations, alpha, modelname):
    if os.path.exists(modelname):
        print(f"Loading model {modelname}...")
        data = np.load(modelname)
        w1 = data["w1"]
        b1 = data["b1"]
        w2 = data["w2"]
        b2 = data["b2"]
        w3 = data["w3"]
        b3 = data["b3"]
    else:
        print(f"Creating new model {modelname}...")
        w1, b1, w2, b2, w3, b3 = init_params()
        
    w1, b1, w2, b2, w3, b3 = train(iterations, alpha, w1, b1, w2, b2, w3, b3)
    np.savez(modelname, w1=w1, b1=b1, w2=w2, b2=b2, w3=w3, b3=b3)
    
def test_model(index, modelname):
    data = np.load(modelname)
    w1 = data["w1"]
    b1 = data["b1"]
    w2 = data["w2"]
    b2 = data["b2"]
    w3 = data["w3"]
    b3 = data["b3"]
    _, _, _, _, _, A3 = forward_prop(X_test, w1, b1, w2, b2, w3, b3)
    test(index, X_test, Y_test, w1, b1, w2, b2, w3, b3)

In [ ]:
train_model(iterations=500, alpha=0.01, modelname="model1.npz")
test_model(np.random.randint(0, X_test.shape[0]), modelname="model1.npz")
# 1341